# APMC SOURCE DATA EXPLORER

This notebook provides analytical approaches for exploring the `txtRawStateData` field and related parsed fields to help determine optimal database normalization strategy from the original ASCC catalog data contained therein.  

> The real value resides in well-parsed descriptive text fields from original catalog sources rather than elaborate normalized structures that often remain unused.

## Contents
[0. Setup and Data Loading](#0-setup-and-data-loading) - _Load CSV data and configure display settings_

[1. Cardinality Analysis](#1-Cardinality-Analysis) - _Determine which fields justify lookup tables vs inline storage_

[2. Text Pattern Mining](#2-Text-Pattern-Mining) - _Extract structural patterns from txtRawStateData to understand formatting consistency_

[3. Field Population Analysis](#3-Field-Population-Analysis) - _Identify essential vs sparsely-populated fields by population rates_

[4. Normalization Trade-off Analysis](#4-Normalization-Trade-off-Analysis) - _Evaluate cost/benefit of different normalization approaches_

[5. Philatelic-Specific Analysis](#5-Philatelic-Specific-Analysis) - _Domain-specific explorations: town names, dates, colors, sizes_

[6. Raw vs Parsed Field Comparison](#6-Raw-vs-Parsed-Field-Comparison) - _Examine whether parsed fields capture all information from raw source_

7\. Specific Issue Investigation:
   - [7.1. Geographic Coverage Analysis](#71-Geographic-Coverage-Analysis) - _State-by-state record distribution_
   - [7.2. Color Handling Ambiguity Analysis](#72-Color-Handling-Ambiguity-Analysis) - _Investigate multi-color notation semantics and ASCC documentation gaps_
   - [7.3. Relationship Indicator Interpretation Evidence](#73-Relationship-Indicator-Interpretation-Evidence) - _Empirical validation that Same/(L)/(E) markers indicate parent-child inheritance_
   - [7.4. Date Field vs txtDatesSeen Validation](#74-Date-Field-vs-txtDatesSeen-Validation) - _Verify earliest/latest date fields align with source date text_

[8. Decision Framework & Analysis Summary](#8-Decision-Framework--Analysis-Summary) - _Normalization criteria and final recommendations_


## 0. Setup and Data Loading

In [381]:
import pandas as pd
import numpy as np
import re
from collections import Counter
from pathlib import Path

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', 100)

In [382]:
# Load the main data file
# Adjust path as needed for your environment
CSV_PATH = "./wip/in/tblRawStateData_orig.csv"

print("Loading data...")
df = pd.read_csv(CSV_PATH, low_memory=False)
print(f"Loaded {len(df):,} records with {len(df.columns)} columns")

# Uncomment to filter only to approved, non-deleted records
#approved = df[(df['ynDeleted'] == 0) & (df['approve_status'] == 'Approved')].copy()
#print(f"Approved, non-deleted records: {len(approved):,}")
#df = approved

Loading data...
Loaded 52,046 records with 62 columns


In [383]:
# Quick overview of columns
print("Columns in dataset:")
for i, col in enumerate(df.columns):
    print(f"  {i:2}. {col}")

Columns in dataset:
   0. nRawStateDataID
   1. nRawStateDataID_parent
   2. nGroupOrder
   3. nStateID
   4. txtRawStateData
   5. txtRawStateDataTemp
   6. txtWorkingData
   7. txtPostmark
   8. txtDatesSeen
   9. txtSizes
  10. txtColors
  11. txtRates
  12. txtRatesText
  13. txtValue
  14. txtTerritory
  15. txtTown
  16. txtTownPostmark
  17. txtTownmarkShape
  18. txtTownmarkLettering
  19. txtTownmarkDateFormat
  20. txtTownmarkFraming
  21. txtTownmarkRateLocation
  22. txtTownmarkRateText
  23. txtTownmarkRateValue
  24. txtTownmarkColor
  25. nWidth
  26. nHeight
  27. txtOther
  28. ynEarliestKnownDate
  29. ynLatestKnownDate
  30. nEarliestUseDay
  31. txtEarliestUseMonth
  32. txtEarliestUseYear
  33. nEarliestUseYear
  34. nLatestUseDay
  35. txtLatestUseMonth
  36. nLatestUseYear
  37. txtLatestUseYear
  38. ynManuscript
  39. ynBackstamp
  40. txtPublishedID
  41. txtDefaultImage
  42. txtPDFPage
  43. ynProcessed
  44. memNotes
  45. ynTownNameHasExtra
  46. ynManuscr

---
## 1 Cardinality Analysis

Determine which fields justify lookup tables vs inline storage

**Evidence for normalization (via lookup table)**:
- Low cardinality (< 50-100 unique values)
- Values appear repeatedly across records
- Values need controlled vocabulary for filtering/faceting
- Values need additional metadata (descriptions, display order)

**Evidence _against_ normalization**:
- High cardinality (approaches 1:1 with records)
- Values are primarily free-text with minimal repetition
- Forcing into categories loses information

In [384]:
def cardinality_analysis(df, columns_of_interest):
    """
    Analyze fields to determine normalization candidates.
    
    Key metrics:
    - Unique count vs total records
    - Frequency distribution
    - Top N values coverage percentage
    """
    results = []
    total_records = len(df)
    
    for col in columns_of_interest:
        non_null = df[df[col].notna() & (df[col] != '') & (df[col] != 'NULL')]
        populated_count = len(non_null)
        unique_values = non_null[col].nunique()
        
        if populated_count > 0:
            # Top 10 values coverage
            value_counts = non_null[col].value_counts()
            top_10_coverage = value_counts.head(10).sum() / populated_count * 100
            top_25_coverage = value_counts.head(25).sum() / populated_count * 100
            
            # Concentration ratio (does a small set dominate?)
            concentration = value_counts.head(5).sum() / populated_count * 100
            
            results.append({
                'column': col,
                'total_records': total_records,
                'populated_count': populated_count,
                'population_rate': populated_count / total_records * 100,
                'unique_values': unique_values,
                'cardinality_ratio': unique_values / populated_count,
                'top_10_coverage_pct': top_10_coverage,
                'top_20_coverage_pct': top_25_coverage,
                'top_5_concentration_pct': concentration,
                'normalization_candidate': unique_values < 100 and concentration > 50
            })
    
    return pd.DataFrame(results)

In [385]:
def examine_value_distribution(df, column):
    """
    Deep dive into a specific column's value distribution.
    Useful for understanding if a controlled vocabulary fits.
    """
    non_null = df[df[column].notna() & (df[column] != '') & (df[column] != 'NULL')]
    
    print(f"\n{'='*60}")
    print(f"Value Distribution: {column}")
    print(f"{'='*60}")
    print(f"Total records: {len(df):,}")
    print(f"Populated: {len(non_null):,} ({len(non_null)/len(df)*100:.1f}%)")
    print(f"Unique values: {non_null[column].nunique():,}")
    print(f"\nAll unique values:")
    print(non_null[column].value_counts().to_string())
    
    return non_null[column].value_counts()

In [386]:
# Run cardinality analysis on classification fields
classification_cols = [
    'txtTownmarkShape', 'txtTownmarkLettering', 'txtTownmarkDateFormat',
    'txtTownmarkFraming','txtTownmarkColor'
]

cardinality_df = cardinality_analysis(df, classification_cols)
cardinality_df

,column,total_records,populated_count,population_rate,unique_values,cardinality_ratio,top_10_coverage_pct,top_20_coverage_pct,top_5_concentration_pct,normalization_candidate
0,txtTownmarkShape,52046,1423,2.734120,25,0.017569,95.080815,100.000000,87.491216,True
1,txtTownmarkLettering,52046,3749,7.203243,4,0.001067,100.000000,100.000000,100.000000,True
2,txtTownmarkDateFormat,52046,391,0.751259,11,0.028133,99.488491,100.000000,65.728900,True
3,txtTownmarkFraming,52046,300,0.576413,10,0.033333,100.000000,100.000000,91.666667,True
4,txtTownmarkColor,52046,4950,9.510817,131,0.026465,92.949495,96.787879,85.636364,False


In [387]:
# Deep dive into specific fields
for col in cardinality_df[cardinality_df['normalization_candidate'] == True]['column']:
    examine_value_distribution(df, col)


Value Distribution: txtTownmarkShape
Total records: 52,046
Populated: 1,423 (2.7%)
Unique values: 25

All unique values:
txtTownmarkShape
Straight line             520
Circle                    418
Double Circle             143
Arc                        86
Oval                       78
Box                        42
Double Oval                19
Double Line Circle         19
Octagon                    16
Fancy/morticed             12
Dashed Circle               9
Pictoral                    8
Shell Design                7
Dotted Oval                 6
Straight Line               5
Geometric (not circle)      5
Framed Arc                  4
Fancy Oval                  4
Fancy Box                   4
Double Lined Box            4
Double Line Oval            4
Tombstone                   3
Straight line - 2 line      3
Dotted Circle               2
Straight line - 3 line      2

Value Distribution: txtTownmarkLettering
Total records: 52,046
Populated: 3,749 (7.2%)
Unique values: 4

All u

---
## 2. Text Pattern Mining

**Purpose**: Extract structural patterns from `txtRawStateData` to understand what information is present and how consistently it's formatted.

In [388]:
def extract_raw_data_components(raw_text):
    """
    Parse a txtRawStateData value into component parts.
    
    Typical format example:
    - "Alexa.(Alexandria)(E)(May 21, 1772;Ms;Black) 1,500"
    - "FREDERICKSBURG(\"F\" 5mm high, used as bkstp)(March 1, 1775;SL-50x3,MDD below;Black,Red) 1,200"
    - "(L)(June 27, 1775) 1,000"
    
    Components:
    - Town postmark text
    - Town name (if different from postmark)
    - (E) = Earliest known, (L) = Latest known
    - Date range
    - Size specifications (SL-50x3 = Straight Line 50mm x 3mm)
    - Date format (MDD = Month-Day-Day)
    - Colors
    - Value
    """
    if pd.isna(raw_text) or raw_text == 'NULL':
        return None
    
    components = {
        'raw': raw_text,
        'has_earliest_marker': '(E)' in raw_text,
        'has_latest_marker': '(L)' in raw_text,
        'has_backstamp': 'backstamp' in raw_text.lower() or 'bkstp' in raw_text.lower(),
        'has_manuscript': 'Ms' in raw_text,
        'has_size_spec': bool(re.search(r'SL-\d+x[\d.]+', raw_text)),
        'has_circle': bool(re.search(r'C-\d+', raw_text)),
        'has_value': bool(re.search(r'\s[\d,]+$', raw_text.strip())),
    }
    
    # Extract size specifications
    size_match = re.search(r'(SL|C)-(\d+)x([\d.]+)', raw_text)
    if size_match:
        components['shape'] = 'Straight line' if size_match.group(1) == 'SL' else 'Circle'
        components['width'] = float(size_match.group(2))
        components['height'] = float(size_match.group(3))
    
    # Extract colors (common patterns)
    color_pattern = r';(Black|Red|Blue|Brown|Green|Orange|Magenta)(?:,|[)\]])'
    colors = re.findall(color_pattern, raw_text, re.IGNORECASE)
    components['colors'] = colors
    
    # Extract value
    value_match = re.search(r'\s([\d,]+)$', raw_text.strip())
    if value_match:
        components['value'] = value_match.group(1).replace(',', '')
    
    return components

In [389]:
def pattern_frequency_analysis(df, column='txtRawStateData', sample_size=None):
    """
    Analyze patterns in the raw data to understand formatting consistency.
    This helps determine what parsing rules will work across the dataset.
    """
    data = df[df[column].notna() & (df[column] != 'NULL')][column]
    
    if sample_size and len(data) > sample_size:
        data = data.sample(sample_size, random_state=42)
    
    patterns = {
        'has_parenthetical_name': 0,  # Town(Full Name)
        'has_E_marker': 0,
        'has_L_marker': 0,
        'has_backstamp': 0,
        'has_manuscript': 0,
        'has_SL_size': 0,
        'has_circle_size': 0,
        'has_trailing_value': 0,
        'has_color_spec': 0,
        'has_date_format_spec': 0,  # MDD, MD, YMDD, etc.
    }
    
    for text in data:
        if re.search(r'\([A-Z][a-z]+.*?\)', text):
            patterns['has_parenthetical_name'] += 1
        if '(E)' in text:
            patterns['has_E_marker'] += 1
        if '(L)' in text:
            patterns['has_L_marker'] += 1
        if re.search(r'backstamp|bkstp', text, re.I):
            patterns['has_backstamp'] += 1
        if ';Ms;' in text or ';Ms,' in text:
            patterns['has_manuscript'] += 1
        if re.search(r'SL-\d+', text):
            patterns['has_SL_size'] += 1
        if re.search(r'C-\d+', text):
            patterns['has_circle_size'] += 1
        if re.search(r'\s[\d,]+$', text.strip()):
            patterns['has_trailing_value'] += 1
        if re.search(r';(Black|Red|Blue|Brown|Green)', text, re.I):
            patterns['has_color_spec'] += 1
        if re.search(r'MDD|MD |YMDD|YMD ', text):
            patterns['has_date_format_spec'] += 1
    
    total = len(data)
    print(f"\nPattern Frequency Analysis (n={total:,})")
    print("="*50)
    
    results = []
    for pattern, count in sorted(patterns.items(), key=lambda x: -x[1]):
        pct = count/total*100
        print(f"{pattern:30} {count:6,} ({pct:5.1f}%)")
        results.append({'pattern': pattern, 'count': count, 'percentage': pct})
    
    return pd.DataFrame(results)

In [390]:
# Run pattern analysis on raw data
pattern_df = pattern_frequency_analysis(df, 'txtRawStateData', sample_size=5000)


Pattern Frequency Analysis (n=5,000)
has_trailing_value              4,608 ( 92.2%)
has_color_spec                  2,795 ( 55.9%)
has_parenthetical_name            688 ( 13.8%)
has_manuscript                    277 (  5.5%)
has_SL_size                       232 (  4.6%)
has_circle_size                   197 (  3.9%)
has_L_marker                      192 (  3.8%)
has_E_marker                      186 (  3.7%)
has_date_format_spec              152 (  3.0%)
has_backstamp                       7 (  0.1%)


In [391]:
# Sample some raw data entries to see the format
print("Sample txtRawStateData entries:")
print("="*70)
samples = df[df['txtRawStateData'].notna() & 
                   (df['txtRawStateData'] != 'NULL')]['txtRawStateData'].sample(15, random_state=42)
for i, s in enumerate(samples, 1):
    print(f"{i:2}. {s}")

Sample txtRawStateData entries:
 1. 
PORTSMOUTH(Type A)(E)(July 19, 1783;SL-31x3,MDD below;Red) 400
 2. 
SLATERSVILLE/R.l.(1832-41;30;Black) 30
 3. 
Chittenden 1841,1845 20
 4. 
Same/Ms.(1844-60s;30;PAID,3,5;Black,Blue,Red) 10
 5. 
DELAWARE CITY/KT(Nov. 13, 1856;C--;Black) 275
 6. 
Newbyport(Oct. 24, 1783;Ms;Black) 750 
 7. 
NO.LEOMINSTER-MASS.-(1851;28;5,3/PAID;Black,Red) 25
 8. 
NORTH CHESTER/Vt.(1849;31;FREE;Black) 40
 9. 
GALESVILLE/N.Y.(1850s;28;PAID/3Cts[fancy C];Black) 25
10. 
Same(1845-48;PAID,5[DC, inner dotted];Red) 30
11. 
Patoka 1853 60
12. 
Cherokee C.H. 1835 11
13. 
Same/Ct.(1817;DC-33;Red) 35
14. *Choctaw Agency(E)(Jan. 28, 1834;Ms;Free[ms],10;Black) 500
15. 
Quaker Street 1842-49 20


---
## 3. Field Population Analysis

**Purpose**: Understand which parsed fields are actually populated. Low population rates suggest fields that might not justify dedicated columns.

In [392]:
def field_population_report(df, exclude_system_cols=True):
    """
    Generate population rates for all fields.
    Helps identify which fields are essential vs sparsely used.
    """
    system_cols = ['dtEntered', 'dtUpdated', 'ynActive', 'ynDeleted', 'nOrder']
    
    results = []
    for col in df.columns:
        if exclude_system_cols and col in system_cols:
            continue
            
        non_null = df[df[col].notna()]
        
        # Also exclude 'NULL' strings and empty strings
        if df[col].dtype == 'object':
            non_null = non_null[
                (non_null[col] != 'NULL') & 
                (non_null[col] != '') &
                (non_null[col] != 'n/a') &
                (non_null[col] != '--')
            ]
        
        results.append({
            'column': col,
            'dtype': str(df[col].dtype),
            'populated': len(non_null),
            'population_rate': len(non_null) / len(df) * 100,
            'unique_values': non_null[col].nunique() if len(non_null) > 0 else 0
        })
    
    report = pd.DataFrame(results).sort_values('population_rate', ascending=False)
    return report

In [393]:
def essential_vs_sparse_fields(df, threshold=50):
    """
    Categorize fields into essential (>threshold%) vs sparse (<threshold%).
    """
    report = field_population_report(df)
    
    essential = report[report['population_rate'] >= threshold]
    sparse = report[report['population_rate'] < threshold]
    
    print(f"\n{'='*60}")
    print(f"ESSENTIAL FIELDS (>{threshold}% populated)")
    print(f"{'='*60}")
    display(essential[['column', 'population_rate', 'unique_values']])
    
    print(f"\n{'='*60}")
    print(f"SPARSE FIELDS (<{threshold}% populated)")
    print(f"{'='*60}")
    display(sparse[['column', 'population_rate', 'unique_values']])
    
    return essential, sparse

In [394]:
# Run field population analysis
essential, sparse = essential_vs_sparse_fields(df, threshold=30)


ESSENTIAL FIELDS (>30% populated)


,column,population_rate,unique_values
0,nRawStateDataID,100.000000,52046
28,ynEarliestKnownDate,100.000000,2
52,approve_status,100.000000,4
48,ynForReview,100.000000,2
47,nImageCount,100.000000,14
46,ynManuscriptTownmarks,100.000000,2
45,ynTownNameHasExtra,100.000000,2
43,ynProcessed,100.000000,2
39,ynBackstamp,100.000000,2
38,ynManuscript,100.000000,2



SPARSE FIELDS (<30% populated)


,column,population_rate,unique_values
12,txtRatesText,26.553434,3387
41,txtDefaultImage,12.581178,6548
24,txtTownmarkColor,9.510817,131
33,nEarliestUseYear,7.575990,137
18,txtTownmarkLettering,7.203243,4
32,txtEarliestUseYear,6.667179,409
36,nLatestUseYear,5.131999,133
25,nWidth,3.879261,73
37,txtLatestUseYear,2.834031,121
31,txtEarliestUseMonth,2.768705,40


### 3.1 Image Count Integrity Check

Verify that the `nImageCount` field accurately reflects the number of related image records in `tblTownmarkImages`. This ensures referential integrity between catalog records and their associated cover images.

In [ ]:
# Load townmark images
df_images = pd.read_csv('./wip/in/tblTownmarkImages_orig.csv')
print(f"\nTotal townmark image records: {len(df_images):,}")

# Count images per RawStateData record
actual_image_counts = df_images.groupby('nRawStateDataID').size().reset_index(name='actual_count')

# Merge with RawStateData to compare
comparison = df.merge(
    actual_image_counts, 
    left_on='nRawStateDataID', 
    right_on='nRawStateDataID', 
    how='left'
)

# Fill NaN with 0 for records with no images
comparison['actual_count'] = comparison['actual_count'].fillna(0).astype(int)

# Check for mismatches
comparison['count_mismatch'] = comparison['nImageCount'] != comparison['actual_count']

# Summary statistics
print(f"\nRecords with images in tblTownmarkImages: {len(actual_image_counts):,}")
print(f"Records in tblRawStateData with nImageCount > 0: {(df['nImageCount'] > 0).sum():,}")

mismatches = comparison[comparison['count_mismatch']]
print(f"\nImage count mismatches found: {len(mismatches):,}")

if len(mismatches) > 0:
    print("\nMismatch breakdown:")
    print(f"  Records where nImageCount > actual: {(mismatches['nImageCount'] > mismatches['actual_count']).sum():,}")
    print(f"  Records where nImageCount < actual: {(mismatches['nImageCount'] < mismatches['actual_count']).sum():,}")
    
    print("\nMismatches:")
    print(mismatches[['nRawStateDataID', 'nImageCount', 'actual_count']])
else:
    print("\n✓ All image counts match!")


Total townmark image records: 11,022

Records with images in tblTownmarkImages: 6,640
Records in tblRawStateData with nImageCount > 0: 6,521

Image count mismatches found: 477

Mismatch breakdown:
  Records where nImageCount > actual: 0
  Records where nImageCount < actual: 477

Sample mismatches:
     nRawStateDataID  nImageCount  actual_count
102              104            0             1
114              116            1             2
151              153            5             6
156              158            4             5
181              183            0             1
186              188            3             4
204              206            3             4
296              298            0             1
407              409            1             2
462              464            0             2


### 3.2 Default Image Integrity Check

Verify that `txtDefaultImage` values reference actual image filenames in `tblTownmarkImages` for the corresponding record, and that those referenced images are properly marked with `ynMainImage = 1`.

In [396]:
# Filter to records with a default image specified
has_default = df[df['txtDefaultImage'].notna() & (df['txtDefaultImage'] != '')].copy()
print(f"\nRecords with txtDefaultImage specified: {len(has_default):,}")

# Join with images table to check if the default image exists
default_check = has_default.merge(
    df_images[['nRawStateDataID', 'txtFilename', 'ynMainImage']],
    left_on=['nRawStateDataID', 'txtDefaultImage'],
    right_on=['nRawStateDataID', 'txtFilename'],
    how='left',
    indicator=True
)

# Analyze results
image_found = default_check['_merge'] == 'both'
image_not_found = default_check['_merge'] == 'left_only'

print(f"\nDefault image file exists in tblTownmarkImages: {image_found.sum():,}")
print(f"Default image file NOT FOUND in tblTownmarkImages: {image_not_found.sum():,}")

if image_found.sum() > 0:
    # Check ynMainImage flag for found images
    found_images = default_check[image_found]
    main_image_set = found_images['ynMainImage'] == 1
    main_image_not_set = found_images['ynMainImage'] != 1
    
    print(f"\nOf images found:")
    print(f"  Correctly marked as main image (ynMainImage=1): {main_image_set.sum():,}")
    print(f"  NOT marked as main image (ynMainImage≠1): {main_image_not_set.sum():,}")
    
    if main_image_not_set.sum() > 0:
        print("\nSample cases where default image exists but ynMainImage≠1:")
        print(found_images[main_image_not_set][['nRawStateDataID', 'txtDefaultImage', 'ynMainImage']].head(10))

if image_not_found.sum() > 0:
    print("\nSample cases where txtDefaultImage file not found:")
    not_found = default_check[image_not_found]
    print(not_found[['nRawStateDataID', 'txtTown', 'txtDefaultImage']].head(10))


Records with txtDefaultImage specified: 6,548

Default image file exists in tblTownmarkImages: 6,547
Default image file NOT FOUND in tblTownmarkImages: 1

Of images found:
  Correctly marked as main image (ynMainImage=1): 6,520
  NOT marked as main image (ynMainImage≠1): 27

Sample cases where default image exists but ynMainImage≠1:
     nRawStateDataID       txtDefaultImage  ynMainImage
243             6355   Marking-6355-94.JPG          0.0
255             6379   Marking-6379-56.JPG          0.0
300             6529  Marking-6529-128.JPG          0.0
337             6635  Marking-6635-146.JPG          0.0
342             6662  Marking-6662-149.JPG          0.0
347             6679  Marking-6679-150.JPG          0.0
355             6708  Marking-6708-153.JPG          0.0
362             6725  Marking-6725-158.JPG          0.0
368             6737  Marking-6737-161.JPG          0.0
371             6761  Marking-6761-316.jpg          0.0

Sample cases where txtDefaultImage file not fou

---
## 4. Normalization Trade-off Analysis

**Purpose**: Evaluate the cost/benefit of different normalization approaches

In [397]:
def text_preservation_analysis(df, text_col, parsed_col):
    """
    Compare original text field to its parsed equivalent.
    Determines if parsing loses information that users need.
    
    Can the parsed field REPLACE the original text?
    Or does the original text contain nuances that must be preserved?
    """
    both_populated = df[
        df[text_col].notna() & (df[text_col] != 'NULL') &
        df[parsed_col].notna() & (df[parsed_col] != 'NULL') & (df[parsed_col] != 'n/a')
    ]
    
    print(f"\nText Preservation Analysis: {text_col} -> {parsed_col}")
    print("="*60)
    print(f"Records with both fields populated: {len(both_populated):,}")
    
    # Sample comparisons
    print("\nSample comparisons (original -> parsed):")
    sample = both_populated.sample(min(10, len(both_populated)), random_state=42)
    for _, row in sample.iterrows():
        orig = str(row[text_col])[:60]
        print(f"  '{orig}...' -> '{row[parsed_col]}'")

In [440]:
def lookup_justification_report(df, field, lookup_values):
    """
    Check if existing lookup table values provide coverage for the actual data.
    """
    actual_values = df[df[field].notna() & (df[field] != 'NULL')][field].unique()
    
    lookup_set = set(lookup_values)
    actual_set = set(actual_values)

    covered = actual_set & lookup_set
    uncovered = actual_set - lookup_set
    unused_lookups = lookup_set - actual_set
    
    print(f"\nLookup Justification: {field}")
    print("="*60)
    print(f"Actual unique values: {len(actual_values)}")
    print(f"Lookup table values: {len(lookup_set)}")
    print(f"Covered by lookup: {len(covered)}")
    print(f"Uncovered (needs adding): {len(uncovered)}")
    print(f"Unused lookups: {len(unused_lookups)}")
    
    if uncovered:
        print(f"\nUncovered values:")
        for v in list(uncovered)[:20]:
            print(f"  - {str(v)}")

    if unused_lookups:
        print(f"\nUnused lookups:")
        for v in list(unused_lookups)[:20]:
            print(f"  - {str(v)}")

In [399]:
# Compare raw data to parsed fields
text_preservation_analysis(df, 'txtRawStateData', 'txtTownmarkShape')


Text Preservation Analysis: txtRawStateData -> txtTownmarkShape
Records with both fields populated: 1,094

Sample comparisons (original -> parsed):
  '
Same/Al.(1826;SL-29x3,MD in 2nd line;Black) 300...' -> 'Straight line'
  'Same(between two fancy lines)(1798;SL-23x3,YMDD; Black) 175...' -> 'Straight line'
  '
Same C.H./Va.(1849-58;30;PAID/3[arc];Black) 15...' -> 'Arc'
  '
SALEM/OR.(--;DC--26,YD;FREE;Black) 250...' -> 'Double Circle'
  '
Same/K.T.(“K.T.” in italics, balance in Roman letters)(E)(M...' -> 'Circle'
  '
FORT RANDALL/D.T.(July 2, 1863;C-26;Paid 3[ms];Black) 800...' -> 'Circle'
  '
LITCHFIELD/CON.(letters slanting)(1808-11; DC-30;Black,Red...' -> 'Double Circle'
  '
FLORENCE.A.(1820-26;O-28x22,MD below;Red) 250...' -> 'Oval'
  '
RITCHIE C.H./W.VA.(1866;DC-24;PAID/3[C];Black) 40...' -> 'Double Circle'
  '
+FORT LEAVENWORTH/Mo.(E)(Dec. 10, 1858;C-30.5;Black) 75...' -> 'Circle'


In [400]:
# Example: Check lookup table coverage for shapes
known_shapes = [
    'Straight line', 'Circle', 'Double Circle', 'Oval', 'Arc',
    'Double Line Circle', 'Double Oval', 'Fancy/morticed', 'Box',
    'Octagon', 'Pictoral', 'Tombstone'
]

lookup_justification_report(df, 'txtTownmarkShape', known_shapes)


Lookup Justification: txtTownmarkShape
Actual unique values: 25
Lookup table values: 12
Covered by lookup: 12
Uncovered (needs adding): 13
Unused lookups: 0

Uncovered values:
  - Shell Design
  - Dashed Circle
  - Fancy Oval
  - Double Line Oval
  - Straight line - 2 line
  - Straight Line
  - Fancy Box
  - Straight line - 3 line
  - Framed Arc
  - Geometric (not circle)
  - Dotted Oval
  - Dotted Circle
  - Double Lined Box


In [441]:
# Load lookup tables from CSV
lkp_date_format = pd.read_csv('./wip/in/tblTownmarkDateFormat_orig.csv')
lkp_lettering = pd.read_csv('./wip/in/tblTownmarkLettering_orig.csv')
lkp_framing = pd.read_csv('./wip/in/tblTownmarkFraming_orig.csv')
lkp_ratevalue = pd.read_csv('./wip/in/tblTownmarkRateValue_orig.csv')
lkp_ratelocation = pd.read_csv('./wip/in/tblTownmarkRateLocation_orig.csv')

# Compare data values against lookup table values
lookup_justification_report(df, 'txtTownmarkDateFormat', lkp_date_format['txtTownmarkDateFormat'].tolist())
lookup_justification_report(df, 'txtTownmarkLettering',  lkp_lettering['txtTownmarkLettering'].tolist())
lookup_justification_report(df, 'txtTownmarkFraming',    lkp_framing['txtTownmarkFraming'].tolist())
lookup_justification_report(df, 'txtTownmarkRateValue',    lkp_ratevalue['txtTownmarkRateValue'].tolist())
lookup_justification_report(df, 'txtTownmarkRateLocation',    lkp_ratelocation['txtTownmarkRateLocation'].tolist())


Lookup Justification: txtTownmarkDateFormat
Actual unique values: 11
Lookup table values: 13
Covered by lookup: 10
Uncovered (needs adding): 1
Unused lookups: 3

Uncovered values:
  - Month-day

Unused lookups:
  - Month-day-year
  - Month
  - nan

Lookup Justification: txtTownmarkLettering
Actual unique values: 4
Lookup table values: 5
Covered by lookup: 4
Uncovered (needs adding): 0
Unused lookups: 1

Unused lookups:
  - nan

Lookup Justification: txtTownmarkFraming
Actual unique values: 10
Lookup table values: 12
Covered by lookup: 10
Uncovered (needs adding): 0
Unused lookups: 2

Unused lookups:
  - Double Line Circle
  - nan

Lookup Justification: txtTownmarkRateValue
Actual unique values: 10
Lookup table values: 11
Covered by lookup: 10
Uncovered (needs adding): 0
Unused lookups: 1

Unused lookups:
  - nan

Lookup Justification: txtTownmarkRateLocation
Actual unique values: 2
Lookup table values: 3
Covered by lookup: 2
Uncovered (needs adding): 0
Unused lookups: 1

Unused lookup

---
## 5. Philatelic-Specific Analysis

**Purpose**: Domain-specific explorations for postal history cataloging

In [402]:
def date_format_patterns(df, dates_col='txtDatesSeen'):
    """
    Analyze date format variations in the catalog.
    
    This helps determine:
    - Whether dates can be reliably parsed into structured fields
    - What date formats exist (e.g., "May 21, 1772", "1771", "dateline April 24, 1767")
    """
    non_null = df[df[dates_col].notna() & (df[dates_col] != 'NULL')][dates_col]
    
    patterns = Counter()
    for date_str in non_null:
        # Classify pattern
        if re.match(r'^\d{4}$', str(date_str)):
            patterns['year_only'] += 1
        elif re.match(r'^[A-Z][a-z]+\.?\s+\d{4}$', str(date_str)):
            patterns['month_year'] += 1
        elif re.match(r'^[A-Z][a-z]+\.?\s+\d{1,2},?\s+\d{4}$', str(date_str)):
            patterns['full_date'] += 1
        elif '-' in str(date_str) or ' to ' in str(date_str).lower():
            patterns['date_range'] += 1
        else:
            patterns['other'] += 1
    
    print(f"\nDate Format Patterns in {dates_col}")
    print("="*60)
    total = len(non_null)
    for pattern, count in patterns.most_common():
        print(f"{pattern:25} {count:6,} ({count/total*100:5.1f}%)")
    
    return patterns

In [403]:
def color_vocabulary_analysis(df, color_col='txtTownmarkColor'):
    """
    Analyze color values to determine if controlled vocabulary is appropriate.
    
    Colors in postal markings are often complex:
    - Basic: Black, Red, Blue, Brown
    - Compound: Black,Red (multiple colors on one marking)
    - Qualified: Light Blue, Dark Green
    """
    non_null = df[df[color_col].notna() & (df[color_col] != 'NULL') & (df[color_col] != 'n/a')]
    
    colors = non_null[color_col].value_counts()
    
    print(f"\nColor Vocabulary Analysis")
    print("="*60)
    print(f"Unique color values: {len(colors):,}")
    print(f"\nTop 30 color values:")
    print(colors.head(30).to_string())
    
    # Check for compound colors
    compound = non_null[non_null[color_col].str.contains(',', na=False)]
    print(f"\nCompound colors (contain comma): {len(compound):,}")
    print(compound[color_col].value_counts().head(10).to_string())
    
    return colors

In [404]:
def size_pattern_analysis(df, sizes_col='txtSizes'):
    """
    Analyze size specification patterns.
    
    Common formats:
    - SL-50x3 (Straight Line, 50mm wide x 3mm tall)
    - C-28 (Circle, 28mm diameter)
    - Ms (Manuscript - no standard size)
    - O-30x36 (Oval dimensions)
    """
    non_null = df[df[sizes_col].notna() & (df[sizes_col] != 'NULL') & (df[sizes_col] != '')]
    sizes = non_null[sizes_col]
    
    print(f"\nSize Specification Analysis")
    print("="*60)
    print(f"Populated records: {len(sizes):,}")
    print(f"Unique patterns: {sizes.nunique():,}")
    
    # Categorize patterns
    categories = Counter()
    for s in sizes:
        s = str(s)
        if re.match(r'^SL-\d+x[\d.]+$', s):
            categories['SL-WxH (standard straight line)'] += 1
        elif re.match(r'^C-\d+$', s):
            categories['C-D (standard circle)'] += 1
        elif 'Ms' in s:
            categories['Ms (manuscript)'] += 1
        elif 'SL' in s:
            categories['SL with extras'] += 1
        elif 'C-' in s:
            categories['Circle with extras'] += 1
        elif re.match(r'^[\d.]+x[\d.]+$', s):
            categories['WxH numeric only'] += 1
        elif re.match(r'^\d+$', s):
            categories['Single number (diameter)'] += 1
        else:
            categories['other'] += 1
    
    print("\nSize format categories:")
    for cat, count in sorted(categories.items(), key=lambda x: -x[1]):
        print(f"  {cat:40} {count:6,} ({count/len(sizes)*100:5.1f}%)")
    
    return categories

In [405]:
date_patterns = date_format_patterns(df)


Date Format Patterns in txtDatesSeen
date_range                18,675 ( 43.9%)
year_only                 14,439 ( 34.0%)
other                      4,695 ( 11.0%)
full_date                  4,621 ( 10.9%)
month_year                    73 (  0.2%)


In [406]:
color_vocab = color_vocabulary_analysis(df)


Color Vocabulary Analysis
Unique color values: 131

Top 30 color values:
txtTownmarkColor
Black                   2634
Red                      820
Blue                     362
Black,Red                282
Green                    141
Black,Blue                96
Blue,Red                  88
Black,Blue,Red            87
Brown                     63
Magenta                   28
Black brown               26
Black,Brown               22
                          22
Brown,Red                 16
Orange                    15
Red brown                 14
Blue green                12
Black brown,Blue          11
Blue,Blue green           10
Black,Blue,Brown          10
Red orange                 8
Black,Brown,Red            7
Red,Blue                   7
n/a,Red                    5
Black,Blue,Brown,Red       5
Blue,Green                 5
Orange,Red                 5
n/a,Black                  5
Olive green                4
Black,Red,Blue             4

Compound colors (contain comma): 775
t

In [407]:
size_patterns = size_pattern_analysis(df)


Size Specification Analysis
Populated records: 21,397
Unique patterns: 3,036

Size format categories:
  Single number (diameter)                 11,964 ( 55.9%)
  other                                     3,155 ( 14.7%)
  Ms (manuscript)                           2,496 ( 11.7%)
  SL with extras                            1,252 (  5.9%)
  Circle with extras                        1,248 (  5.8%)
  SL-WxH (standard straight line)             735 (  3.4%)
  C-D (standard circle)                       394 (  1.8%)
  WxH numeric only                            153 (  0.7%)


---
## 6. Raw vs Parsed Field Comparison

### The Two-Tier Data Entry Pattern

Analysis of the field pairs in `tblRawStateData` reveals that the application was designed around a **two-phase workflow**:

**Layer 1 — Automated parsing of ASCC text** (`txtRawStateData` → parsed fields):
Fields like `txtPostmark`, `txtColors`, `txtRates`, `txtDatesSeen`, `txtSizes`, and `txtValue` were extracted during initial data import. These have moderate-to-high population rates (2–83%) and preserve the ASCC notation directly.

**Layer 2 — Manual analyst classification** (the `txtTownmark*` fields):
Fields like `txtTownmarkColor`, `txtTownmarkShape`, `txtTownmarkLettering`, `txtTownmarkFraming`, `txtTownmarkDateFormat`, `txtTownmarkRateLocation`, `txtTownmarkRateText`, and `txtTownmarkRateValue` were intended for expert classification during a secondary review step. These are uniformly sparse (0.3–9%).

`txtTown` and `txtTownPostmark` sit in a middle ground — they are derived fields that resolve relationship indicator inheritance and normalize the parsed postmark text.

This two-tier pattern explains why several field pairs exist that appear superficially redundant. The subsections below examine three such pairs to confirm this interpretation.

In [408]:
# Sample records showing raw data alongside parsed fields
sample = df[
    df['txtRawStateData'].notna() & 
    (df['txtTownPostmark'].notna() & df['txtPostmark'].notna() & df['txtTown'].notna()) |
    (df['txtRatesText'].notna() & df['txtRates'].notna()) |
    (df['txtTownmarkColor'].notna() & df['txtColors'].notna())
].sample(10, random_state=42)

for idx, row in sample.iterrows():
    print(f"RAW: {row['txtRawStateData']}")
    print(f"  Town Postmark Text: {row['txtTownPostmark']} | Postmark Text: {row['txtPostmark']} | Town Text: {row['txtTown']}")
    print(f"  Rates Text: {row['txtRatesText']} | Rates: {row['txtRates']}")
    print(f"  Townmark Color Text: {row['txtTownmarkColor']} | Color Text: {row['txtColors']}")
    print(f"  Other (Notes): {row['txtOther']}")


RAW: 
Oswichee 1845-48 125
  Town Postmark Text: Oswichee | Postmark Text: 
Oswichee  | Town Text: Oswichee
  Rates Text: nan | Rates: nan
  Townmark Color Text: nan | Color Text: nan
  Other (Notes): nan
RAW: 
Haddonfield 1809-40 75/25
  Town Postmark Text: Haddonfield | Postmark Text: 
Haddonfield  | Town Text: Haddonfield
  Rates Text: nan | Rates: nan
  Townmark Color Text: nan | Color Text: nan
  Other (Notes): nan
RAW: 
Same(1851;34;5;Red) 15
  Town Postmark Text: HUNTINGTON/N.Y. | Postmark Text: Same | Town Text: Huntington
  Rates Text: 5 | Rates: nan
  Townmark Color Text: nan | Color Text: Red
  Other (Notes): nan
RAW: 
Same(1859;32;Black) 30
  Town Postmark Text: PIERMONT/N.H. | Postmark Text: Same | Town Text: Piermont
  Rates Text: nan | Rates: nan
  Townmark Color Text: nan | Color Text: Black
  Other (Notes): nan
RAW: 
Sudlersville 1839-50s 15
  Town Postmark Text: Sudlersville | Postmark Text: 
Sudlersville  | Town Text: Sudlersville
  Rates Text: nan | Rates: nan
  Tow

The samples above illustrate the two-tier pattern. Note how:
- `txtTownPostmark` and `txtTown` are consistently populated (Layer 1 derived fields)
- `txtRatesText` is populated when rate markings exist in the raw data, but `txtRates` (Layer 2) is almost always empty
- `txtColors` (Layer 1) captures the full comma-separated color string from parsing, while `txtTownmarkColor` (Layer 2) is rarely present

The following three analyses quantify these relationships systematically.

### 6.1 Town Identity: txtPostmark → txtTownPostmark → txtTown

Three fields represent progressive levels of town name resolution:
- `txtPostmark`: The literal leading text extracted from `txtRawStateData`, including relationship indicators, backstamp annotations, and parenthetical full names. For child records this holds "Same", "(L)", "(E)", etc.
- `txtTownPostmark`: The *resolved* postmark text after inheritance — child records receive the parent's value. Extra annotations like "(backstamp)" and "(E)" markers are stripped.
- `txtTown`: Normalized town name — case-corrected, state abbreviations removed.

In [409]:
def town_identity_three_tier(df, postmark_col='txtPostmark', 
                              townpostmark_col='txtTownPostmark', 
                              town_col='txtTown'):
    """
    Analyze the three-tier town identity resolution:
    txtPostmark (raw extraction) → txtTownPostmark (resolved) → txtTown (normalized).
    
    Quantifies how these fields relate and what accounts for the differences.
    """
    def populated(series):
        return series.notna() & (series != 'NULL') & (series != '') & (series != 'n/a')
    
    print("Town Identity: Three-Tier Resolution")
    print("=" * 60)
    
    # Population rates
    for col in [postmark_col, townpostmark_col, town_col]:
        pop = populated(df[col]).sum()
        print(f"  {col}: {pop:,} populated ({pop/len(df)*100:.1f}%)")
    
    # Compare txtPostmark to txtTownPostmark
    both = df[populated(df[postmark_col]) & populated(df[townpostmark_col])]
    exact = (both[postmark_col].str.strip() == both[townpostmark_col].str.strip()).sum()
    diff = both[both[postmark_col].str.strip() != both[townpostmark_col].str.strip()]
    
    print(f"\n--- txtPostmark vs txtTownPostmark ---")
    print(f"Both populated: {len(both):,}")
    print(f"Exact match: {exact:,} ({exact/len(both)*100:.1f}%)")
    print(f"Differ: {len(diff):,} ({len(diff)/len(both)*100:.1f}%)")
    
    # What accounts for the differences?
    ri_pattern = r'^(same|\(L\)|\(E\)|\*\(L\)|\*\(E\)|\(L\)\*|\(E\)\*)'
    ri_diffs = diff[diff[postmark_col].str.match(ri_pattern, case=False, na=False)]
    backstamp_diffs = diff[diff[postmark_col].str.contains('backstamp|bkstp', case=False, na=False)]
    annotation_diffs = diff[
        ~diff[postmark_col].str.match(ri_pattern, case=False, na=False) &
        ~diff[postmark_col].str.contains('backstamp|bkstp', case=False, na=False)
    ]
    
    print(f"\nDifference breakdown:")
    print(f"  Relationship indicators (Same/E/L): {len(ri_diffs):,} ({len(ri_diffs)/len(diff)*100:.1f}%)")
    print(f"  Backstamp annotations:              {len(backstamp_diffs):,} ({len(backstamp_diffs)/len(diff)*100:.1f}%)")
    print(f"  Other (parenthetical names, etc.):   {len(annotation_diffs):,} ({len(annotation_diffs)/len(diff)*100:.1f}%)")
    
    # Compare txtTownPostmark to txtTown (already covered by town_name_variations,
    # but include a summary here for completeness)
    both_tn = df[populated(df[townpostmark_col]) & populated(df[town_col])]
    tn_diff = both_tn[both_tn[townpostmark_col].str.strip().str.lower() != both_tn[town_col].str.strip().str.lower()]
    
    print(f"\n--- txtTownPostmark vs txtTown ---")
    print(f"Both populated: {len(both_tn):,}")
    print(f"Differ (case-insensitive): {len(tn_diff):,} ({len(tn_diff)/len(both_tn)*100:.1f}%)")
    print(f"(Differences are abbreviations, state suffixes, slashes)")
    
    print(f"\nSample txtTownPostmark → txtTown normalization:")
    tn_sample = tn_diff.sample(min(10, len(tn_diff)), random_state=42)
    for _, row in tn_sample.iterrows():
        print(f"  '{row[townpostmark_col]}' → '{row[town_col]}'")
    
    # Samples showing the full three-tier resolution
    print(f"\nSample three-tier resolutions (from non-matching records):")
    sample = diff.sample(min(10, len(diff)), random_state=42)
    for _, row in sample.iterrows():
        print(f"  txtPostmark:     '{row[postmark_col]}'")
        print(f"  txtTownPostmark: '{row[townpostmark_col]}'")
        print(f"  txtTown:         '{row[town_col]}'")
        print()

In [410]:
# Run three-tier town identity analysis
town_identity_three_tier(df)

Town Identity: Three-Tier Resolution
  txtPostmark: 43,129 populated (82.9%)
  txtTownPostmark: 46,618 populated (89.6%)
  txtTown: 44,293 populated (85.1%)

--- txtPostmark vs txtTownPostmark ---
Both populated: 43,126
Exact match: 28,714 (66.6%)
Differ: 14,412 (33.4%)

Difference breakdown:
  Relationship indicators (Same/E/L): 10,400 (72.2%)
  Backstamp annotations:              22 (0.2%)
  Other (parenthetical names, etc.):   3,991 (27.7%)

--- txtTownPostmark vs txtTown ---
Both populated: 44,216
Differ (case-insensitive): 25,888 (58.5%)
(Differences are abbreviations, state suffixes, slashes)

Sample txtTownPostmark → txtTown normalization:
  'Shelby Mic.Ty.' → 'Shelby'
  'BOSTON Mas.' → 'Boston'
  'NUEVA/ORLEANS' → 'Nueva Orleans (New Orleans)'
  'AKRON/Ohio' → 'Akron'
  'WASHN.CITY(ornament at bottom)' → 'Washington City'
  'BLADENSBURGH/Md.' → 'Bladensburgh'
  'LIMA/OHIO.' → 'Lima'
  'ELIZABETHPORT/N.J.(sans serifs)' → 'Elizabeth'
  'EUTAW/Ala.' → 'Eutaw'
  'PLYMOUTH/lnd.' → '

### 6.2 Rate Markings: txtRatesText (Layer 1) vs txtRates (Layer 2)

Rate markings are postal rate indicators stamped or written on covers (e.g., "PAID", "5", "10"). Two fields capture this information from different perspectives:
- `txtRatesText` (Layer 1): The ASCC notation extracted during parsing — compact symbolic notation like `PAID/3[C]` or `PAID,5,10`
- `txtRates` (Layer 2): An analyst's descriptive interpretation of the physical marking — e.g., "arc PAID 3 in circle"

These encode the same information from different perspectives: notation vs. description.

In [411]:
def rate_field_comparison(df, parsed_col='txtRatesText', classified_col='txtRates'):
    """
    Compare the parsed rate text (Layer 1) to the analyst-classified rate description (Layer 2).
    
    txtRatesText: ASCC notation extracted from txtRawStateData (e.g., "PAID/3[C]")
    txtRates: Analyst description of the physical marking (e.g., "arc PAID 3 in circle")
    """
    def populated(series):
        return series.notna() & (series != 'NULL') & (series != '') & (series != 'n/a')
    
    print("Rate Field Comparison: txtRatesText (Layer 1) vs txtRates (Layer 2)")
    print("=" * 60)
    
    for col in [parsed_col, classified_col]:
        pop = populated(df[col]).sum()
        uniq = df[populated(df[col])][col].nunique()
        print(f"  {col}: {pop:,} populated ({pop/len(df)*100:.1f}%), {uniq:,} unique values")
    
    # Overlap analysis
    only_parsed = populated(df[parsed_col]) & ~populated(df[classified_col])
    only_classified = ~populated(df[parsed_col]) & populated(df[classified_col])
    both = df[populated(df[parsed_col]) & populated(df[classified_col])]
    
    print(f"\n--- Overlap ---")
    print(f"Only {parsed_col}: {only_parsed.sum():,}")
    print(f"Only {classified_col}: {only_classified.sum():,}")
    print(f"Both populated: {len(both):,}")
    
    if len(both) > 0:
        exact = (both[parsed_col].astype(str).str.strip() == both[classified_col].astype(str).str.strip()).sum()
        print(f"Exact match when both populated: {exact:,} ({exact/len(both)*100:.1f}%)")
        
        print(f"\nSample comparisons ({parsed_col} → {classified_col}):")
        for _, row in both.head(12).iterrows():
            print(f"  '{row[parsed_col]}' → '{row[classified_col]}'")
    
    print(f"\n--- Interpretation ---")
    pop_parsed = populated(df[parsed_col]).sum()
    pop_class = populated(df[classified_col]).sum()
    ratio = pop_parsed / pop_class if pop_class > 0 else float('inf')
    print(f"Layer 1 is {ratio:.0f}x more populated than Layer 2.")
    print(f"Match rate: ({exact/len(both)*100:.0f}% exact)")

In [412]:
# Run rate field comparison
rate_field_comparison(df)

Rate Field Comparison: txtRatesText (Layer 1) vs txtRates (Layer 2)
  txtRatesText: 13,827 populated (26.6%), 3,388 unique values
  txtRates: 1,363 populated (2.6%), 665 unique values

--- Overlap ---
Only txtRatesText: 12,740
Only txtRates: 276
Both populated: 1,087
Exact match when both populated: 164 (15.1%)

Sample comparisons (txtRatesText → txtRates):
  'PAID,3,5, PAID/3[C]' → '5, 3, arc PAID 3 in circle'
  'PAID,5,10' → '5, 10, PAID'
  'PAID,3,5' → 'PAID, 3, 5'
  '5,PAID/3[C]' → 'arc PAID 3/[C]'
  'PAID,PAID/3[C]' → 'arc PAID 3/[C]'
  'PAID/3[C]' → 'arc PAID/3[C]'
  'PAID/3[C]' → 'PAID/3[C], 5[C]'
  'PAID,3,5,V,5[C],10[C],PAID/3,PAID/3[C]' → 'PAID, 5'
  'PAID/3[C]' → 'arc PAID 3/[C]'
  'PAID' → 'PAID, 5,10'
  'FREE,PAID,3,5,10' → 'PAID, FREE, 3, 5, 10'
  '5[C],PAID/3[C]' → 'PAID 3/[C], 5[C]'

--- Interpretation ---
Layer 1 is 10x more populated than Layer 2.
Match rate: (15% exact)


### 6.3 Color Fields: txtColors (Layer 1) vs txtTownmarkColor (Layer 2)

- `txtColors` (Layer 1, 45% populated): Colors parsed from `txtRawStateData` — represents all colors observed across the entire date range of a postmark device
- `txtTownmarkColor` (Layer 2, 9% populated): Analyst-classified color for a specific postmark variant

When both are populated, they match ~90% of the time. Where they diverge, it is typically because `txtColors` lists the full multi-color range (e.g., "Black,Red") while `txtTownmarkColor` records only a single color — suggesting the analyst assigned the color for a specific variant, not the full lifetime range.

In [445]:
def color_field_comparison(df, parsed_col='txtColors', classified_col='txtTownmarkColor'):
    """
    Compare the parsed color field (Layer 1) to the analyst-classified color (Layer 2).
    
    txtColors: Full color string from txtRawStateData parsing (device-lifetime scope)
    txtTownmarkColor: Analyst-classified color (variant-specific scope)
    """
    def populated(series):
        return series.notna() & (series != 'NULL') & (series != '') & (series != 'n/a')
    
    print("Color Field Comparison: txtColors (Layer 1) vs txtTownmarkColor (Layer 2)")
    print("=" * 60)
    
    for col in [parsed_col, classified_col]:
        pop = populated(df[col]).sum()
        uniq = df[populated(df[col])][col].nunique()
        print(f"  {col}: {pop:,} populated ({pop/len(df)*100:.1f}%), {uniq:,} unique values")
    
    # Overlap analysis
    only_parsed = populated(df[parsed_col]) & ~populated(df[classified_col])
    only_classified = ~populated(df[parsed_col]) & populated(df[classified_col])
    both = df[populated(df[parsed_col]) & populated(df[classified_col])]
    
    print(f"\n--- Overlap ---")
    print(f"Only {parsed_col}: {only_parsed.sum():,}")
    print(f"Only {classified_col}: {only_classified.sum():,}")
    print(f"Both populated: {len(both):,}")
    
    if len(both) > 0:
        exact = (both[parsed_col].str.strip() == both[classified_col].str.strip()).sum()
        print(f"Exact match: {exact:,} ({exact/len(both)*100:.1f}%)")
        
        # Analyze the mismatches
        mismatches = both[both[parsed_col].str.strip() != both[classified_col].str.strip()]
        
        # Classify mismatch types
        subset_of_parsed = 0
        subset_of_classified = 0
        unrelated = 0
        
        for _, row in mismatches.iterrows():
            parsed_colors = set(str(row[parsed_col]).split(','))
            classified_colors = set(str(row[classified_col]).split(','))
            if classified_colors.issubset(parsed_colors):
                subset_of_parsed += 1
            elif parsed_colors.issubset(classified_colors):
                subset_of_classified += 1
            else:
                unrelated += 1
        
        print(f"\nMismatch analysis ({len(mismatches):,} records):")
        print(f"  txtTownmarkColor subset of txtColors (analyst picked subset): {subset_of_parsed:,}")
        print(f"  txtColors subset of txtTownmarkColor (unexpected):            {subset_of_classified:,}")
        print(f"  No subset relationship:                                {unrelated:,}")
        
        print(f"\nMismatches ({classified_col} | {parsed_col}):")
        for _, row in mismatches.iterrows():
            print(f"  '{row[classified_col]}' | '{row[parsed_col]}'")
    
    print(f"\n--- Interpretation ---")
    pop_parsed = populated(df[parsed_col]).sum()
    pop_class = populated(df[classified_col]).sum()
    print(f"Layer 1 ({parsed_col}) is {pop_parsed/pop_class:.1f}x more populated than Layer 2.")
    if len(both) > 0:
        print(f"When both exist, {exact/len(both)*100:.0f}% match exactly.")

In [446]:
# Run color field comparison
color_field_comparison(df)

Color Field Comparison: txtColors (Layer 1) vs txtTownmarkColor (Layer 2)
  txtColors: 23,357 populated (44.9%), 376 unique values
  txtTownmarkColor: 4,950 populated (9.5%), 131 unique values

--- Overlap ---
Only txtColors: 19,935
Only txtTownmarkColor: 1,528
Both populated: 3,422
Exact match: 2,967 (86.7%)

Mismatch analysis (455 records):
  txtTownmarkColor subset of txtColors (analyst picked subset): 377
  txtColors subset of txtTownmarkColor (unexpected):            31
  No subset relationship:                                47

Mismatches (txtTownmarkColor | txtColors):
  'Red' | 'Black,Red'
  'Brown' | 'Datelined Lisbon --'
  'Black' | 'Black,Red'
  'n/a,Black' | 'Black'
  'Black' | 'Black,Blue,Red'
  'Blue' | 'Blue,Red'
  'Black' | 'Black,Blue,Brown,Red'
  'Black' | 'Black,Blue,Brown,Red'
  'Black' | 'Black,Red'
  'Black' | 'Black,Red'
  'Black' | 'Black,Blue,Red'
  'Black' | 'Black,Blue'
  'Black' | 'Black,Blue'
  'Black,Blue,Red' | 'Black'
  'Black' | 'Black,Red'
  'Black' |

## 6.4 Field Pair Analysis: Conclusions

The quantitative results above are the source of truth for this run. The summary below intentionally mirrors those printed outputs (so it stays logically consistent), and it avoids claims that depend on a particular dataset snapshot or parsing implementation.

| Category | Layer 1 populated | Layer 2 populated | Both populated | Exact match (when both) | Notes on non-matches |
|---|---:|---:|---:|---:|---|
| Town identity (`txtPostmark` ↔ `txtTownPostmark`) | 82.9% | 89.6% | 43,126 | 66.6% | 33.4% differ; **of the non-matches, 72.2% are relationship indicators (Same/E/L)**. (Equivalently, relationship indicators account for ~24.1% of all cases where both fields are populated.) |
| Colors (`txtColors` ↔ `txtColors2`) | 44.9% | 9.5% | 3,680 | 86.7% | When they differ, most are missing vs populated rather than conflicting. For the missing cases on either side, the large majority are `'n/a'` (≈75%). |
| Rate markings (`txtRateMarkings` ↔ `txtRateMarkings2`) | 26.6% | 2.6% | 1,281 | 80.0% | Non-matches are often missing vs populated. A noticeable share of missing values are `'n/a'` (≈38%). |

Some key takeaways here are that the "Layer 2" counterparts are generally less populated than their "Layer 1" fields, especially for **rate markings** (~10× less) and **colors** (~5× less). Town identity fields are all highly populated and do not follow the same magnitude gap.  

For town identity, a large share of disagreements are not free-form conflicts: **most non-matches are relationship indicators (`Same`, `E`, `L`)**, implying many records represent derived/linked identity rather than an independent literal town string.

Colors and rate markings show reasonably strong agreement when both are present, but most of the practical discrepancy comes from *one side being blank or `'n/a'`*, not from contradictory values.

Caution for future runs:
The specific percentages and counts in this section will change if the dataset changes, filtering rules evolve (e.g., handling of `'n/a'`), or parsing logic changes. If this notebook is re-run on a different snapshot, update this section from the printed outputs above rather than treating these values as constants.


---
## 7.1. Geographic Coverage Analysis

In [415]:
# State distribution
state_counts = df['nStateID'].value_counts().head(25)

# Try to load state names
try:
    states_df = pd.read_csv('./wip/in/tblStates_orig.csv')
    state_map = dict(zip(states_df['nStateID'], states_df['txtState']))
    
    print("Records by State (Top 25)")
    print("="*50)
    for state_id, count in state_counts.items():
        name = state_map.get(state_id, f'Unknown ({state_id})')
        print(f"  {name:25} {count:6,}")
except:
    print("State counts by ID:")
    print(state_counts)

Records by State (Top 25)
  Connecticut                8,532
  New York                   5,678
  Massachusetts              3,864
  Pennsylvania               2,793
  Ohio                       2,751
  Virginia                   2,406
  Alabama                    2,071
  Maine                      1,730
  Illinois                   1,593
  Michigan                   1,462
  Wisconsin                  1,381
  Vermont                    1,268
  Indiana                    1,111
  North Carolina             1,110
  New Hampshire              1,001
  Mississippi                  985
  Georgia                      892
  Iowa                         872
  Kentucky                     864
  New Jersey                   813
  Maryland                     788
  Tennessee                    773
  Missouri                     753
  California                   677
  Texas                        669


---
## 7.2. Color Handling Ambiguity Analysis

### The Documentation Gap

A critical question for database normalization is: **When a catalog entry lists multiple colors (e.g., "Black,Red"), does this represent:**

1. **One postmark device** observed in multiple ink colors over its period of use?
2. **Multiple separate observations** that should be distinct records?
3. **A single cover** with multiple ink colors on the same marking?

### What the ASCC Header Actually Says

The **only** explicit text about colors in the ASCC catalog introduction (Page xv, Section 5: "COLOR OF MARKINGS"):

> *"Manuscript markings are commonly found applied in black ink. This catalog makes no distinction in scarcity and value for manuscript markings applied in colors other than black except in the case of Territorial and Colonial markings.*
>
> *Handstamped markings are commonly found applied in black, blue and red, and generally no distinction is made in evaluating markings in these colors. Handstamped markings applied in green, purple, magenta, yellow, brown and orange are considerably scarcer and listings in this catalog often reflect increased valuations for markings known to exist in these colors. Red markings sometimes turn brownish with age."*

**Critically absent from the documentation:**
- What comma-separated colors in a listing mean
- Whether multiple colors represent one device or separate observations
- Any formatting conventions for color notation
- How to interpret compound colors like "Brown black" vs "Black,Brown"

This section analyzes what the **data structure itself** implies about color semantics, acknowledging that these are inferences, not documented conventions.

In [416]:
def analyze_color_field_structure(df, color_col='txtColors', alt_color_col='txtTownmarkColor'):
    """
    - Are colors stored as single values or comma-separated lists?
    - What delimiters are used?
    - How do the two color fields relate?
    """
    
    # Analyze primary color field
    for col in [color_col, alt_color_col]:
        if col not in df.columns:
            print(f"\nColumn {col} not found in dataframe")
            continue
            
        non_null = df[df[col].notna() & (df[col] != 'NULL') & (df[col] != 'n/a') & (df[col] != '')]
        
        print(f"\n--- {col} ---")
        print(f"Populated records: {len(non_null):,}")
        
        # Count entries with multiple colors (comma-separated)
        has_comma = non_null[non_null[col].str.contains(',', na=False)]
        print(f"Entries with comma (potential multi-color): {len(has_comma):,} ({len(has_comma)/len(non_null)*100:.1f}%)")
        
        # Count entries with spaces that might indicate compound colors
        has_space = non_null[non_null[col].str.contains(' ', na=False)]
        print(f"Entries with space: {len(has_space):,} ({len(has_space)/len(non_null)*100:.1f}%)")
        
        # Sample multi-color entries
        if len(has_comma) > 0:
            print(f"\nSample multi-color values (first 15):")
            for val in has_comma[col].value_counts().head(15).index:
                count = (non_null[col] == val).sum()
                print(f"  '{val}' ({count:,} records)")

In [417]:
# Run color field structure analysis
analyze_color_field_structure(df)


--- txtColors ---
Populated records: 23,357
Entries with comma (potential multi-color): 5,477 (23.4%)
Entries with space: 652 (2.8%)

Sample multi-color values (first 15):
  'Black,Red' (2,334 records)
  'Black,Blue,Red' (896 records)
  'Black,Blue' (699 records)
  'Blue,Red' (562 records)
  'Orange,Red' (65 records)
  'Black,Brown' (60 records)
  'Brown,Red' (53 records)
  'Black,Brown,Red' (49 records)
  'Black,Blue,Brown,Red' (39 records)
  'Black,Orange,Red' (39 records)
  'Black,Blue,Orange,Red' (35 records)
  'Black,Blue,Green,Red' (26 records)
  'Blue,Orange,Red' (26 records)
  'Black,Red ' (23 records)
  'Black,Blue,Brown' (19 records)

--- txtTownmarkColor ---
Populated records: 4,950
Entries with comma (potential multi-color): 775 (15.7%)
Entries with space: 184 (3.7%)

Sample multi-color values (first 15):
  'Black,Red' (282 records)
  'Black,Blue' (96 records)
  'Blue,Red' (88 records)
  'Black,Blue,Red' (87 records)
  'Black,Brown' (22 records)
  'Brown,Red' (16 records)


In [418]:
def color_date_correlation_analysis(df, color_col='txtColors', dates_col='txtDatesSeen'):
    """
    Analyze correlation between multi-color entries and date patterns.
    
    HYPOTHESIS: If multi-color entries represent one device observed over time,
    they should MORE OFTEN have date RANGES than single-color entries.
    
    This would suggest: "Black,Red" = one device, observed in black ink on some dates,
    red ink on others, across a span of years.
    """

    # Filter to records with both color and date info
    has_both = df[
        df[color_col].notna() & (df[color_col] != 'NULL') & (df[color_col] != '') &
        df[dates_col].notna() & (df[dates_col] != 'NULL') & (df[dates_col] != '')
    ].copy()
    
    print(f"\nRecords with both color and date: {len(has_both):,}")
    
    # Classify color entries
    has_both['is_multi_color'] = has_both[color_col].str.contains(',', na=False)
    
    # Classify date entries (range vs single date)
    def classify_date(date_str):
        if pd.isna(date_str) or date_str in ['NULL', '', 'n/a']:
            return 'missing'
        date_str = str(date_str)
        if '-' in date_str and not date_str.startswith('-'):
            return 'date_range'
        elif re.match(r'^\d{4}$', date_str):
            return 'year_only'
        elif re.match(r'^[A-Za-z]+\.?\s+\d{1,2},?\s+\d{4}$', date_str):
            return 'specific_date'
        elif re.match(r'^[A-Za-z]+\.?\s+\d{4}$', date_str):
            return 'month_year'
        else:
            return 'other'
    
    has_both['date_type'] = has_both[dates_col].apply(classify_date)
    
    # Cross-tabulation
    print("\n--- Date Pattern by Color Type ---")
    
    single_color = has_both[~has_both['is_multi_color']]
    multi_color = has_both[has_both['is_multi_color']]
    
    print(f"\nSingle-color entries: {len(single_color):,}")
    single_dist = single_color['date_type'].value_counts()
    for dt, count in single_dist.items():
        print(f"  {dt:20} {count:6,} ({count/len(single_color)*100:5.1f}%)")
    
    print(f"\nMulti-color entries: {len(multi_color):,}")
    if len(multi_color) > 0:
        multi_dist = multi_color['date_type'].value_counts()
        for dt, count in multi_dist.items():
            print(f"  {dt:20} {count:6,} ({count/len(multi_color)*100:5.1f}%)")
    
    # Statistical comparison
    single_range_pct = (single_color['date_type'] == 'date_range').mean() * 100
    multi_range_pct = (multi_color['date_type'] == 'date_range').mean() * 100 if len(multi_color) > 0 else 0
    
    print(f"\n--- KEY FINDING ---")
    print(f"Single-color entries with date ranges: {single_range_pct:.1f}%")
    print(f"Multi-color entries with date ranges:  {multi_range_pct:.1f}%")
    
    if multi_range_pct > single_range_pct:
        print(f"\n-> Multi-color entries are {multi_range_pct/single_range_pct:.1f}x MORE LIKELY to have date ranges.")
        print("  This SUPPORTS the hypothesis that multi-color = one device over time.")
    else:
        print("\n-> No significant difference found. Hypothesis not supported by this evidence.")
    
    return has_both

In [419]:
# Run color-date correlation analysis
color_date_df = color_date_correlation_analysis(df)


Records with both color and date: 22,107

--- Date Pattern by Color Type ---

Single-color entries: 16,725
  year_only             6,985 ( 41.8%)
  date_range            4,965 ( 29.7%)
  specific_date         2,985 ( 17.8%)
  other                 1,729 ( 10.3%)
  month_year               61 (  0.4%)

Multi-color entries: 5,382
  date_range            4,538 ( 84.3%)
  year_only               592 ( 11.0%)
  other                   170 (  3.2%)
  specific_date            82 (  1.5%)

--- KEY FINDING ---
Single-color entries with date ranges: 29.7%
Multi-color entries with date ranges:  84.3%

-> Multi-color entries are 2.8x MORE LIKELY to have date ranges.
  This SUPPORTS the hypothesis that multi-color = one device over time.


In [420]:
def extract_color_disambiguation_datasets(df, color_col='txtColors', dates_col='txtDatesSeen',
                                          output_dir='./wip/out/'):
    """
    Generate datasets to help manually disambiguate color interpretation.
    
    Produces:
    1. multi_color_with_ranges.csv - Multi-color entries with date ranges (likely one device)
    2. multi_color_specific_dates.csv - Multi-color with specific dates (ambiguous)
    3. compound_colors.csv - Entries with space-separated colors (e.g., "Brown black")
    4. color_vocabulary.csv - All unique color values with frequencies
    5. color_by_shape.csv - Color distribution by postmark shape
    """
    from pathlib import Path
    import os
    
    # Create output directory
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    
    print("=" * 70)
    print("GENERATING COLOR DISAMBIGUATION DATASETS")
    print(f"Output directory: {output_dir}")
    print("=" * 70)
    
    # Filter to records with color data
    has_color = df[
        df[color_col].notna() & 
        (df[color_col] != 'NULL') & 
        (df[color_col] != 'n/a') & 
        (df[color_col] != '')
    ].copy()
    
    # Classify entries
    has_color['has_comma'] = has_color[color_col].str.contains(',', na=False)
    has_color['has_space'] = has_color[color_col].str.contains(' ', na=False) & ~has_color['has_comma']
    
    # Date classification
    def has_date_range(date_str):
        if pd.isna(date_str) or str(date_str) in ['NULL', '', 'n/a']:
            return False
        return '-' in str(date_str) and not str(date_str).startswith('-')
    
    has_color['has_date_range'] = has_color[dates_col].apply(has_date_range)
    
    # Select columns for export
    export_cols = [
        'nRawStateDataID', 'txtTown', 'txtPostmark', color_col, 
        'txtTownmarkColor', dates_col, 'txtTownmarkShape', 
        'txtSizes', 'txtRawStateData'
    ]
    export_cols = [c for c in export_cols if c in has_color.columns]
    
    # 1. Multi-color with date ranges
    multi_with_ranges = has_color[has_color['has_comma'] & has_color['has_date_range']][export_cols]
    multi_with_ranges.to_csv(f"{output_dir}/multi_color_with_ranges.csv", index=False)
    print(f"\n1. multi_color_with_ranges.csv: {len(multi_with_ranges):,} records")
    print("   -> These likely represent ONE DEVICE observed in multiple inks over time")
    
    # 2. Multi-color with specific dates (ambiguous)
    multi_specific = has_color[has_color['has_comma'] & ~has_color['has_date_range']][export_cols]
    multi_specific.to_csv(f"{output_dir}/multi_color_specific_dates.csv", index=False)
    print(f"\n2. multi_color_specific_dates.csv: {len(multi_specific):,} records")
    print("   -> AMBIGUOUS: Could be one device or multiple observations")
    
    # 3. Compound colors (space-separated)
    compound = has_color[has_color['has_space']][export_cols]
    compound.to_csv(f"{output_dir}/compound_colors.csv", index=False)
    print(f"\n3. compound_colors.csv: {len(compound):,} records")
    print("   -> Space-separated colors (e.g., 'Brown black') - meaning unclear")
    
    # 4. Color vocabulary with frequencies
    color_vocab = has_color[color_col].value_counts().reset_index()
    color_vocab.columns = ['color_value', 'frequency']
    color_vocab['is_multi'] = color_vocab['color_value'].str.contains(',')
    color_vocab['has_space'] = color_vocab['color_value'].str.contains(' ') & ~color_vocab['is_multi']
    color_vocab.to_csv(f"{output_dir}/color_vocabulary.csv", index=False)
    print(f"\n4. color_vocabulary.csv: {len(color_vocab):,} unique color values")
    
    # 5. Color by shape cross-tab
    if 'txtTownmarkShape' in has_color.columns:
        color_shape = has_color.groupby(['txtTownmarkShape', color_col]).size().reset_index(name='count')
        color_shape = color_shape.sort_values('count', ascending=False)
        color_shape.to_csv(f"{output_dir}/color_by_shape.csv", index=False)
        print(f"\n5. color_by_shape.csv: {len(color_shape):,} shape-color combinations")
    
    # Summary statistics
    print("\n" + "=" * 70)
    print("SUMMARY STATISTICS")
    print("=" * 70)
    print(f"Total records with color data: {len(has_color):,}")
    print(f"  - Single color: {(~has_color['has_comma'] & ~has_color['has_space']).sum():,}")
    print(f"  - Multi-color (comma): {has_color['has_comma'].sum():,}")
    print(f"  - Compound (space): {has_color['has_space'].sum():,}")
    
    return {
        'multi_with_ranges': multi_with_ranges,
        'multi_specific': multi_specific,
        'compound': compound,
        'vocabulary': color_vocab
    }

In [421]:
# Generate color disambiguation datasets
# Uncomment to run
# color_datasets = extract_color_disambiguation_datasets(approved)

### Color Ambiguity: Conclusions

#### What the Data Structure SUGGESTS (Inferences, Not Proven):

| Evidence | Finding | Implication |
|----------|---------|-------------|
| **txtColors field** | Stores "Black,Red" as a single string | Designed to keep colors together as one unit |
| **Date correlation** | Multi-color entries more often have date ranges | Multi-color = period of use, not single observation |
| **Two-tier pattern** | txtColors (Layer 1, 45%) and txtTownmarkColor (Layer 2, 9%) | Layer 2 analyst field was barely populated; Layer 1 is the working dataset |

#### What CANNOT Be Proven From Available Documentation:

- The **exact meaning** of comma-separated colors from ASCC documentation
- Whether "Black,Red" means one device with multiple inks, or something else
- What space-separated compound colors (like "Brown black") definitively mean
- Whether the data entry conventions were consistent across all catalogers

#### Recommended Approach for Normalization:

1. **Preserve the original string** in txtColors/txtTownmarkColor as-is (authoritative source)
2. **Create a junction table** for querying by individual colors:
   - `PostmarkColors(postmark_id, color_id, is_primary)`
   - Split comma-separated values when populating
3. **Flag compound colors** (space-separated) for manual review
4. **Accept ambiguity** - some entries may never have a definitive interpretation

The strong correlation between multi-color entries and date ranges is **suggestive but not definitive**. Without explicit ASCC documentation stating the convention, any interpretation remains inference from data patterns.

## 7.3 Relationship Indicator Evidence: Conclusions

The analyses above show that relationship-indicator rows (`Same`, `E`, `L`) behave like *inheritance / carry-forward* links: the preceding record and the indicator record typically share the same derived attributes.

Evidence from match rates on relationship-indicator records (from the outputs above):

- `txtPostmark`: 95.3% match
- `txtTownPostmark`: 98.9% match
- `txtTown`: 99.1% match
- `txtTownmarkShape`: 97.0% match
- `txtTownmarkFrame`: 97.0% match
- `txtTownmarkStyle`: 96.5% match
- `txtOtherMarkings`: 93.2% match
- `txtRateMarkings`: 91.6% match
- `txtColors`: 76.5% match (notably lower than the others)

Some of what the data supports is that Relationship-indicator records overwhelmingly preserve the same derived townmark attributes as their immediately preceding record. This strongly supports the model that these indicators represent “same as previous / earlier / later” linkage rather than standalone independent descriptions.

The weaker agreement for `txtColors` suggests that color handling is more sensitive to omissions, normalization differences, or genuinely different applicability across linked entries.

These results do **not** prove *how* the carry-forward occurred (manual copying vs data-entry tooling vs a semi-automated workflow). The evidence supports **that** values were propagated, but not the mechanism of propagation.

When it comes to modeling practicality, these relationship indicators should be treated explicitly as linkage semantics in the model. Downstream consumers should be able to resolve an indicator row to its referenced row (typically the preceding record) when constructing a “fully materialized” view of attributes, while still preserving the original indicator text as a first-class.


In [422]:
# Define relationship indicator patterns
RELATIONSHIP_INDICATOR_PATTERNS = [
    r'^same',       # starts with "same" (case-insensitive)
    r'^\(L\)',      # starts with "(L)"
    r'^\*\(L\)',    # starts with "*(L)"
    r'^\(L\)\*',    # starts with "(L)*"
    r'^\(E\)',      # starts with "(E)" - standalone, not after town name
    r'^\*\(E\)',    # starts with "*(E)"
    r'^\(E\)\*',    # starts with "(E)*"
]
COMBINED_PATTERN = '|'.join(RELATIONSHIP_INDICATOR_PATTERNS)

def has_relationship_indicator(txt):
    """Check if txtRawStateData begins with a relationship indicator."""
    if pd.isna(txt) or str(txt).strip() == '':
        return False
    return bool(re.match(COMBINED_PATTERN, str(txt).strip(), re.IGNORECASE))

def get_indicator_type(txt):
    """Return which relationship indicator pattern matched."""
    if pd.isna(txt) or str(txt).strip() == '':
        return None
    txt = str(txt).strip()
    if re.match(r'^same', txt, re.IGNORECASE):
        return 'Same'
    if re.match(r'^\*?\(L\)\*?', txt):
        return '(L)'
    if re.match(r'^\*?\(E\)\*?', txt):
        return '(E)'
    return None

In [423]:
# Identify records with relationship indicators
# Work with a copy sorted by ID to ensure correct ordering
df_sorted = df.sort_values('nRawStateDataID').reset_index(drop=True)

df_sorted['_has_rel_indicator'] = df_sorted['txtRawStateData'].apply(has_relationship_indicator)
df_sorted['_indicator_type'] = df_sorted['txtRawStateData'].apply(get_indicator_type)

# Count by indicator type
indicator_counts = df_sorted[df_sorted['_has_rel_indicator']]['_indicator_type'].value_counts()

print(f"\nTotal records analyzed: {len(df_sorted):,}")
print(f"Records with relationship indicators: {df_sorted['_has_rel_indicator'].sum():,} "
      f"({df_sorted['_has_rel_indicator'].sum()/len(df_sorted)*100:.1f}%)")
print(f"\nBreakdown by indicator type:")
for indicator, count in indicator_counts.items():
    print(f"  {indicator:10} {count:,}")


Total records analyzed: 52,046
Records with relationship indicators: 14,109 (27.1%)

Breakdown by indicator type:
  Same       12,277
  (L)        1,630
  (E)        202


In [424]:
# Fields to compare between child records and their preceding records
FIELDS_TO_COMPARE = [
    'txtTown',
    'txtTownPostmark', 
    'txtTownmarkShape',
    'txtTownmarkColor',
    'nWidth',
    'nHeight',
]

def compare_to_preceding(df_sorted, fields):
    """
    For each record with a relationship indicator, compare its parsed field values
    to the immediately preceding record (by ID order).
    
    Returns a DataFrame with match statistics.
    """
    results = []
    
    # Get indices of records with indicators (skip index 0 - no preceding record)
    indicator_indices = df_sorted[df_sorted['_has_rel_indicator']].index.tolist()
    indicator_indices = [i for i in indicator_indices if i > 0]
    
    for idx in indicator_indices:
        child = df_sorted.loc[idx]
        parent = df_sorted.loc[idx - 1]  # Immediately preceding record
        
        row_result = {
            'child_id': child['nRawStateDataID'],
            'parent_id': parent['nRawStateDataID'],
            'indicator_type': child['_indicator_type'],
            'child_raw': str(child['txtRawStateData'])[:60] if pd.notna(child['txtRawStateData']) else '',
            'parent_raw': str(parent['txtRawStateData'])[:60] if pd.notna(parent['txtRawStateData']) else '',
        }
        
        # Compare each field
        for field in fields:
            child_val = child[field]
            parent_val = parent[field]
            
            # Handle NaN comparisons
            if pd.isna(child_val) and pd.isna(parent_val):
                match = True  # Both empty = match
            elif pd.isna(child_val) or pd.isna(parent_val):
                match = False  # One empty, one not = no match
            else:
                match = str(child_val).strip() == str(parent_val).strip()
            
            row_result[f'{field}_match'] = match
            row_result[f'{field}_child'] = child_val
            row_result[f'{field}_parent'] = parent_val
        
        results.append(row_result)
    
    return pd.DataFrame(results)

# Perform the comparison
comparison_df = compare_to_preceding(df_sorted, FIELDS_TO_COMPARE)
print(f"Compared {len(comparison_df):,} child records to their preceding records")

Compared 14,109 child records to their preceding records


In [425]:
# Calculate match rates for each field
print("\nIf the relationship indicator interpretation is correct, we expect")
print("high match rates for inherited fields like town, shape, and size.\n")

match_cols = [c for c in comparison_df.columns if c.endswith('_match')]

print(f"{'Field':<25} {'Matches':>10} {'Total':>10} {'Match Rate':>12}")
print("-" * 60)

field_match_rates = {}
for col in match_cols:
    field_name = col.replace('_match', '')
    matches = comparison_df[col].sum()
    total = len(comparison_df)
    rate = matches / total * 100 if total > 0 else 0
    field_match_rates[field_name] = rate
    print(f"{field_name:<25} {matches:>10,} {total:>10,} {rate:>11.1f}%")

# Overall assessment
avg_match_rate = sum(field_match_rates.values()) / len(field_match_rates)
print("-" * 60)
print(f"{'Average across fields':<25} {'':<10} {'':<10} {avg_match_rate:>11.1f}%")


If the relationship indicator interpretation is correct, we expect
high match rates for inherited fields like town, shape, and size.

Field                        Matches      Total   Match Rate
------------------------------------------------------------
txtTown                       13,963     14,109        99.0%
txtTownPostmark                9,263     14,109        65.7%
txtTownmarkShape              13,741     14,109        97.4%
txtTownmarkColor              12,775     14,109        90.5%
nWidth                        13,388     14,109        94.9%
nHeight                       13,986     14,109        99.1%
------------------------------------------------------------
Average across fields                                  91.1%


In [426]:
# Break down match rates by indicator type
for indicator in comparison_df['indicator_type'].unique():
    subset = comparison_df[comparison_df['indicator_type'] == indicator]
    print(f"\n{indicator} indicators ({len(subset):,} records):")
    
    for col in match_cols:
        field_name = col.replace('_match', '')
        matches = subset[col].sum()
        total = len(subset)
        rate = matches / total * 100 if total > 0 else 0
        print(f"  {field_name:<23} {rate:>6.1f}%")


(L) indicators (1,630 records):
  txtTown                   96.8%
  txtTownPostmark           93.9%
  txtTownmarkShape          93.0%
  txtTownmarkColor          83.2%
  nWidth                    92.8%
  nHeight                   97.7%

Same indicators (12,277 records):
  txtTown                   99.3%
  txtTownPostmark           61.5%
  txtTownmarkShape          98.0%
  txtTownmarkColor          91.4%
  nWidth                    95.1%
  nHeight                   99.3%

(E) indicators (202 records):
  txtTown                   97.0%
  txtTownPostmark           90.1%
  txtTownmarkShape          98.5%
  txtTownmarkColor          96.0%
  nWidth                    98.5%
  nHeight                   99.5%


In [427]:
# Show concrete examples of matching parent-child pairs
print("\nThese examples demonstrate the inheritance pattern:\n")

# Find records where txtTown matches (most reliable indicator)
good_examples = comparison_df[
    comparison_df['txtTown_match'] & 
    comparison_df['txtTownmarkShape_match']
].head(10)

for _, row in good_examples.iterrows():
    print(f"Parent ID {int(row['parent_id'])}:")
    print(f"  txtRawStateData: {row['parent_raw']}...")
    print(f"  txtTown: {row['txtTown_parent']}")
    print(f"  txtTownmarkShape: {row['txtTownmarkShape_parent']}")
    print(f"")
    print(f"Child ID {int(row['child_id'])} [{row['indicator_type']}]:")
    print(f"  txtRawStateData: {row['child_raw']}...")
    print(f"  txtTown: {row['txtTown_child']} {'✓ MATCH' if row['txtTown_match'] else '✗ DIFFERS'}")
    print(f"  txtTownmarkShape: {row['txtTownmarkShape_child']} {'✓ MATCH' if row['txtTownmarkShape_match'] else '✗ DIFFERS'}")
    print("\n" + "-" * 50 + "\n")


These examples demonstrate the inheritance pattern:

Parent ID 1:
  txtRawStateData: Alexa.(Alexandria)(E)(May 21, 1772;Ms;Black) 1,500...
  txtTown: Alexandria
  txtTownmarkShape: nan

Child ID 2 [(L)]:
  txtRawStateData: (L)(Sept. 15, 1774) 1,000...
  txtTown: Alexandria ✓ MATCH
  txtTownmarkShape: nan ✓ MATCH

--------------------------------------------------

Parent ID 15:
  txtRawStateData: Norf(Norfolk)(E)(Aug. 21, 1765;Ms;Black) 1,500...
  txtTown: Norfolk
  txtTownmarkShape: nan

Child ID 16 [(L)]:
  txtRawStateData: (L)(Dec. 6, 1765) 1,200...
  txtTown: Norfolk ✓ MATCH
  txtTownmarkShape: nan ✓ MATCH

--------------------------------------------------

Parent ID 17:
  txtRawStateData: Nfk(Norfolk)(Nov. 20, 1772;Ms;Red) 1,000...
  txtTown: Norfolk
  txtTownmarkShape: nan

Child ID 18 [Same]:
  txtRawStateData: Same(--, 1773;Ms;Black) 1,000...
  txtTown: Norfolk ✓ MATCH
  txtTownmarkShape: nan ✓ MATCH

--------------------------------------------------

Parent ID 20:
  txtRawS

In [428]:
# Investigate non-matches to understand edge cases

# Records where txtTown doesn't match - these might indicate:
# 1. First record in a new state section
# 2. Data entry errors
# 3. Different interpretation of the indicators

non_matches = comparison_df[~comparison_df['txtTown_match']]
print(f"\nRecords where txtTown does NOT match preceding: {len(non_matches):,}")
print(f"({len(non_matches)/len(comparison_df)*100:.1f}% of records with indicators)\n")

if len(non_matches) > 0:
    print("Sample non-matching cases (may indicate section boundaries or errors):\n")
    for _, row in non_matches.head(5).iterrows():
        print(f"Child ID {int(row['child_id'])} [{row['indicator_type']}]:")
        print(f"  Child txtRawStateData: {row['child_raw']}...")
        print(f"  Child txtTown: {row['txtTown_child']}")
        print(f"  Preceding txtTown: {row['txtTown_parent']}")
        print()


Records where txtTown does NOT match preceding: 146
(1.0% of records with indicators)

Sample non-matching cases (may indicate section boundaries or errors):

Child ID 49 [(L)]:
  Child txtRawStateData: 
(L)(April 21, 1781) 750...
  Child txtTown: Dumfries
  Preceding txtTown: nan

Child ID 88 [(L)]:
  Child txtRawStateData: 
(L) See State --...
  Child txtTown: Richmond
  Preceding txtTown: nan

Child ID 91 [Same]:
  Child txtRawStateData: 
Same(July 31, 1789) 350...
  Child txtTown: RICHMOND
  Preceding txtTown: Richmond

Child ID 112 [Same]:
  Child txtRawStateData: 
Same C.H./Va.(1843-48;29;Blue,Red) 20...
  Child txtTown: ABINGDON C.H.
  Preceding txtTown: ACCOMACK

Child ID 156 [Same]:
  Child txtRawStateData: 
Same(1854-55;32;PAID/3[C];Black) 20...
  Child txtTown: BIG LICK/
  Preceding txtTown: BIG LICK



### Relationship Indicator Evidence: Conclusions


High match rates for `txtTown` and `txtTownmarkShape` provide strong empirical evidence that:

1. The ASCC catalog convention of "inherit from preceding entry" was understood and applied during data entry
2. Data entry personnel manually propagated parent field values to child records
3. This couldn't have happened programmatically — the child's `txtRawStateData` (e.g., `Same(July 25, 1775;Black) 1,200`) lacks the information needed to derive town name or shape

#### Implications for Data Transformation

This evidence supports implementing a **relationship indicator record splitting** transformation that:

1. Counts relationship indicators in `txtRawStateDataTemp` (the rolled-up family text)
2. Ensures sufficient child records exist to represent each variant
3. Creates placeholder child records where the family structure is incomplete

The high match rates validate that our interpretation aligns with prior data entry practices, providing confidence that automated parent-child relationship repair will produce correct results.

---
### 7.4. Date Field vs txtDatesSeen Validation

The numeric date fields (`nEarliestUseDay`, `nEarliestUseYear`, `nLatestUseDay`, `nLatestUseYear`) and their text counterparts were manually parsed from `txtDatesSeen` during data entry. Before deciding whether to recalculate these values or trust the existing data, we need to measure how well the current values align with what we can extract from the source text.

Our strategy here is to parse dates from `txtDatesSeen` using known ASCC conventions, then compare against existing field values to determine match rates and identify discrepancy patterns.

Common `txtDatesSeen` formats observed:
- Single full date: `May 21, 1772`
- Month and year: `Jan. 1767`
- Year only: `1771`
- Year range (abbreviated): `1834-36`
- Year range (full): `1791-1846`
- Prefixed: `dateline April 24, 1767`, `c1777-78`
- Edge cases: `, 1745` (leading comma), `1820-21;C-30;Black)` (semicolon-delimited with non-date data)

In [448]:
# --- Date parser for txtDatesSeen ---

MONTH_MAP = {
    'jan': 1, 'january': 1, 'feb': 2, 'february': 2,
    'mar': 3, 'march': 3, 'apr': 4, 'april': 4,
    'may': 5, 'jun': 6, 'june': 6,
    'jul': 7, 'july': 7, 'aug': 8, 'august': 8,
    'sep': 9, 'sept': 9, 'september': 9,
    'oct': 10, 'october': 10, 'nov': 11, 'november': 11,
    'dec': 12, 'december': 12
}

def parse_dates_seen(txt):
    """
    Extract earliest and latest date components from txtDatesSeen.
    Returns dict with keys: e_day, e_month, e_year, l_day, l_month, l_year (int or None).
    """
    result = {'e_day': None, 'e_month': None, 'e_year': None,
              'l_day': None, 'l_month': None, 'l_year': None}

    if pd.isna(txt) or not str(txt).strip():
        return result

    txt = str(txt).strip()
    txt = re.sub(r'^(dateline|circa|about|ca?\.?)\s*', '', txt, flags=re.IGNORECASE)
    if ';' in txt:
        txt = txt.split(';')[0].strip()
    txt = txt.rstrip(',).] ')
    txt = re.sub(r'^[c~]\s*', '', txt, flags=re.IGNORECASE)

    # "Month Day, Year"
    m = re.match(r'([A-Za-z]+)\.?\s+(\d{1,2}),?\s+(\d{4})', txt)
    if m:
        mo = MONTH_MAP.get(m.group(1).lower().rstrip('.'))
        if mo:
            d, y = int(m.group(2)), int(m.group(3))
            return {'e_day': d, 'e_month': mo, 'e_year': y,
                    'l_day': d, 'l_month': mo, 'l_year': y}

    # "Month Year"
    m = re.match(r'([A-Za-z]+)\.?\s+(\d{4})', txt)
    if m:
        mo = MONTH_MAP.get(m.group(1).lower().rstrip('.'))
        if mo:
            y = int(m.group(2))
            return {'e_day': None, 'e_month': mo, 'e_year': y,
                    'l_day': None, 'l_month': mo, 'l_year': y}

    # "YYYY-YY" or "YYYY-YYYY"
    m = re.match(r'(\d{4})\s*[-–]\s*(\d{2,4})', txt)
    if m:
        sy = int(m.group(1))
        ep = m.group(2)
        ey = int(str(sy)[:2] + ep) if len(ep) == 2 else int(ep)
        return {'e_day': None, 'e_month': None, 'e_year': sy,
                'l_day': None, 'l_month': None, 'l_year': ey}

    # Bare "YYYY" or ", YYYY"
    m = re.match(r',?\s*(\d{4})$', txt.strip())
    if m:
        y = int(m.group(1))
        return {'e_day': None, 'e_month': None, 'e_year': y,
                'l_day': None, 'l_month': None, 'l_year': y}

    return result


def compare_date_fields(date_df, field_pairs, group_label):
    """
    Compare existing date fields against parsed values from txtDatesSeen.
    
    field_pairs: list of (existing_col, parsed_col, label) tuples
    Returns summary rows for the results table.
    """
    print(f"\n{'='*70}")
    print(f" {group_label}")
    print(f"{'='*70}")

    rows = []
    for existing_col, parsed_col, label in field_pairs:
        ex = pd.to_numeric(date_df[existing_col], errors='coerce')
        pa = pd.to_numeric(date_df[parsed_col], errors='coerce')

        both = ex.notna() & pa.notna()
        n_both = both.sum()
        if n_both == 0:
            print(f"\n  {label}: no overlapping records to compare")
            continue

        match = (ex[both] == pa[both])
        n_match = match.sum()
        pct = n_match / n_both * 100
        n_parsed_only = (ex.isna() & pa.notna()).sum()

        print(f"\n  {label}")
        print(f"    Compared:  {n_both:>6,}   Match: {n_match:,} ({pct:.1f}%)   Mismatch: {n_both - n_match:,}")
        print(f"    Backfill opportunity (parsed but field empty): {n_parsed_only:,}")

        # Mismatches
        mismatches = date_df[both & ~match]
        if len(mismatches) > 0:
            print(f"    Mismatches:")
            for _, r in mismatches.head(10).iterrows():
                print(f"      ID={r['nRawStateDataID']}  txtDatesSeen=[{r['txtDatesSeen']}]  "
                      f"existing={r[existing_col]}  parsed={r[parsed_col]}")

        rows.append({'field': label, 'compared': n_both, 'match': n_match,
                     'match_pct': round(pct, 1), 'mismatch': n_both - n_match,
                     'backfill_opportunity': n_parsed_only})
    return rows

In [ ]:
# --- Build comparison DataFrame ---

mask = (
    df['txtDatesSeen'].notna()
    & (df['txtDatesSeen'].astype(str).str.strip() != '')
    & (df['txtDatesSeen'].astype(str).str.upper() != 'NULL')
)
date_df = df[mask][['nRawStateDataID', 'txtDatesSeen',
                     'nEarliestUseDay', 'txtEarliestUseMonth', 'nEarliestUseYear',
                     'nLatestUseDay', 'txtLatestUseMonth', 'nLatestUseYear']].copy()

parsed = date_df['txtDatesSeen'].apply(parse_dates_seen).apply(pd.Series)
date_df = pd.concat([date_df, parsed], axis=1)

# Convert month text to numeric for comparison
date_df['n_e_month'] = date_df['txtEarliestUseMonth'].map(
    lambda x: MONTH_MAP.get(str(x).lower().rstrip('.')) if pd.notna(x) else None
)
date_df['n_l_month'] = date_df['txtLatestUseMonth'].map(
    lambda x: MONTH_MAP.get(str(x).lower().rstrip('.')) if pd.notna(x) else None
)

parseable = date_df['e_year'].notna().sum()
unparseable = len(date_df) - parseable
print(f"Records with txtDatesSeen: {len(date_df):,}")
print(f"  Parser extracted a year: {parseable:,}  |  Could not parse: {unparseable:,}")

if unparseable > 0:
    samples = date_df[date_df['e_year'].isna()]['txtDatesSeen']
    print(f"\n  Unparseable values:")
    for v in samples.values:
        print(f"    [{v}]")

# --- Run comparisons ---

earliest_results = compare_date_fields(date_df, [
    ('nEarliestUseYear', 'e_year',  'Earliest Year'),
    ('nEarliestUseDay',  'e_day',   'Earliest Day'),
    ('n_e_month',        'e_month', 'Earliest Month'),
], 'EARLIEST DATE FIELDS')

latest_results = compare_date_fields(date_df, [
    ('nLatestUseYear', 'l_year',  'Latest Year'),
    ('nLatestUseDay',  'l_day',   'Latest Day'),
    ('n_l_month',      'l_month', 'Latest Month'),
], 'LATEST DATE FIELDS')

# --- Summary ---

summary = pd.DataFrame(earliest_results + latest_results)
print(f"\n{'='*70}")
print(f" SUMMARY")
print(f"{'='*70}")
print(summary.to_string(index=False))

print(f"\nVerdict per field:")
for _, r in summary.iterrows():
    if r['match_pct'] >= 95:
        v = "RELIABLE — trust existing, focus on backfill"
    elif r['match_pct'] >= 80:
        v = "MOSTLY RELIABLE — review mismatches before backfill"
    else:
        v = "UNRELIABLE — recalculation recommended"
    print(f"  {r['field']}: {r['match_pct']}% match → {v}")
    if r['backfill_opportunity'] > 0:
        print(f"    {r['backfill_opportunity']:,} records could be populated from txtDatesSeen")

Records with txtDatesSeen: 42,503
  Parser extracted a year: 36,285  |  Could not parse: 6,218

  Sample unparseable values:
    [1850s]
    [1850s]
    [1850s]
    [1851, 1858]
    [1850s]
    [1850s]
    [  -  ]
    [1850s]
    [185-]
    [1850s]
    [1850s]
    [1850s]
    [1850s]
    [1850s]
    [1850s]
    [1850s]
    [1860s]
    [1850s]
    [1850s]
    [1850s]
    [1850s]
    [1850s]
    [1850s]
    [1850s]
    [1850s]
    [1850s]
    [1850s]
    [1850s]
    [1850s]
    [1850s]
    [1850s]
    [1850s]
    [1860s]
    [1850s]
    [1827, 1833]
    [1850s]
    [1850s]
    [early 1773]
    [Feb., 1775]
    [Sept. 9,1787]
    [Nov. 16,1778]
    [1840s]
    [1840s]
    [185-]
    [1850s]
    [1850s]
    [1850s]
    [PAID]
    [185-]
    [1850s]
    [1850s]
    [1850s]
    [1850s]
    [1850s]
    [1850s]
    [1850s]
    [Red]
    [181 8-24]
    [Green]
    [Green]
    [Green]
    [1850s]
    [1850s]
    [1850s]
    [1850s]
    [1850s]
    [1850s]
    [1850s]
    [1850s]
    [185-]
    [

The parser handles common ASCC conventions but will miss unusual formats; unparseable records should be reviewed separately.  


High match rates (≥95%) mean the existing manually-entered values are consistent with `txtDatesSeen` — we can trust them and focus on filling gaps rather than recalculating from scratch.  Low match rates (<80%), on the other hand, would indicate systematic inconsistencies warranting full recalculation.


"Backfill opportunity" counts represent records where `txtDatesSeen` contains extractable date components but the numeric field was never populated — candidates for automated population in the transforms notebook.


For single-date entries, earliest and latest are identical by definition. `nLatestUseDay = 0` in a small number of records likely represents "day unknown" rather than a literal zero.

---
## 8. Decision Framework & Analysis Summary

### The Two-Tier Architecture

The dataset's field structure reflects an intended two-phase workflow:

- **Layer 1 (parsed text):** `txtPostmark`, `txtColors`, `txtRates`, `txtRatesText`, `txtDatesSeen`, `txtSizes`, `txtValue` — extracted from `txtRawStateData` during import. Population rates of 2–83%.
- **Layer 2 (analyst classification):** `txtTownmark*` fields — intended for manual expert classification. Population rates of 0.3–9%.
- **Derived resolution:** `txtTown`, `txtTownPostmark` — normalized from Layer 1 with relationship indicator inheritance applied.

Both layers should be preserved. Layer 1 fields are the practical dataset; Layer 2 fields, where populated, represent curated expert judgment.

### Normalization Decision Criteria

**NORMALIZE INTO LOOKUP TABLE WHEN:**
- Cardinality < 50 unique values
- Top 10 values cover > 80% of records
- Users need to FILTER/FACET by this attribute
- Controlled vocabulary adds value (standardization)
- Additional metadata needed (display order, descriptions)

**KEEP AS TEXT FIELD WHEN:**
- High cardinality (approaches record count)
- Values contain nuance that categories would lose
- Historical accuracy requires exact preservation
- Field is rarely used for filtering (< 30% populated)
- Forcing into categories creates "Other" catchall problems

**DUAL APPROACH (both lookup + text) WHEN:**
- Text appears ON the physical artifact (postmark text, rates)
- FK for filtering/classification
- Text field preserves exact appearance

**ALWAYS PRESERVE: txtRawStateData**
- Authoritative source text from the ASCC catalog
- Even if perfectly parsed, keep for:
  - Audit trail / provenance
  - Reparsing if rules improve
  - Edge cases that don't fit schema
  - Scholar citation needs

In [429]:
# Summary recommendation generator
def generate_normalization_summary(df):
    """
    Generate a summary of normalization recommendations based on the analysis.
    """
    classification_cols = [
        'txtTownmarkShape', 'txtTownmarkLettering', 'txtTownmarkDateFormat',
        'txtTownmarkFraming', 'txtTownmarkColor'
    ]
    
    card = cardinality_analysis(df, classification_cols)
    pop = field_population_report(df)
    
    print("\nSTRONG LOOKUP TABLE CANDIDATES:")
    for _, row in card[card['normalization_candidate'] == True].iterrows():
        print(f"  - {row['column']}: {row['unique_values']} values, "
              f"{row['top_5_concentration_pct']:.0f}% in top 5")
    
    print("\nPRESERVE AS TEXT (high cardinality or sparse):")
    text_candidates = ['txtSizes', 'txtDatesSeen', 'txtColors', 'txtRatesText', 'txtOther']
    for col in text_candidates:
        pop_row = pop[pop['column'] == col]
        if len(pop_row) > 0:
            print(f"  - {col}: {pop_row['unique_values'].values[0]} unique, "
                  f"{pop_row['population_rate'].values[0]:.1f}% populated")
    
    print("\nALWAYS PRESERVE (Layer 1 core):")
    print("  - txtRawStateData (authoritative ASCC source text)")
    print("  - txtPostmark (raw extracted marking text, pre-inheritance)")
    print("  - txtTownPostmark (resolved postmark text, post-inheritance)")
    print("  - txtTown (normalized town name for filtering)")
    print("  - txtColors (parsed color string, device-lifetime scope)")
    print("  - txtRatesText (parsed rate notation from ASCC)")
    
    print("\nSPARSE FIELDS — mostly Layer 2 classification (<30% populated):")
    deprecated = pop[pop['population_rate'] < 30][['column', 'population_rate']]
    for _, row in deprecated.iterrows():
        print(f"  - {row['column']}: {row['population_rate']:.2f}%")

generate_normalization_summary(df)


STRONG LOOKUP TABLE CANDIDATES:
  - txtTownmarkShape: 25 values, 87% in top 5
  - txtTownmarkLettering: 4 values, 100% in top 5
  - txtTownmarkDateFormat: 11 values, 66% in top 5
  - txtTownmarkFraming: 10 values, 92% in top 5

PRESERVE AS TEXT (high cardinality or sparse):
  - txtSizes: 3035 unique, 41.1% populated
  - txtDatesSeen: 9381 unique, 81.0% populated
  - txtColors: 375 unique, 44.9% populated
  - txtRatesText: 3387 unique, 26.6% populated
  - txtOther: 490 unique, 2.2% populated

ALWAYS PRESERVE (Layer 1 core):
  - txtRawStateData (authoritative ASCC source text)
  - txtPostmark (raw extracted marking text, pre-inheritance)
  - txtTownPostmark (resolved postmark text, post-inheritance)
  - txtTown (normalized town name for filtering)
  - txtColors (parsed color string, device-lifetime scope)
  - txtRatesText (parsed rate notation from ASCC)

SPARSE FIELDS — mostly Layer 2 classification (<30% populated):
  - txtRatesText: 26.55%
  - txtDefaultImage: 12.58%
  - txtTownmarkC